In [1]:
!pip install ragas langchain-openai langchain datasets pandas

In [3]:
import os
import pandas as pd
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall

# 🔑 Set your API key if it's not already in your system environment variables
# If you have it globally set, you can comment these two lines out.
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "<Enter your APIs>"

print("🔄 Step 1: Initializing the AI Judge (GPT-4o)...")
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# --- THE TEST BASELINE ---
print("📦 Step 2: Ingesting Logs for Strategy A (Basic Fixed Character Chunking)...")
strategy_a_logs = {
    "user_input": ["What are the rules regarding commercial racing and when must an accident be reported?"],
    "contexts": [["Our company premium insurance plan rejects any vehicle claims involving unauthorized commercial racing."]], 
    "response": ["The policy states that claims involving unauthorized commercial racing will be rejected. I couldn't find information about an accident reporting deadline."],
    "reference": ["Our company premium insurance plan rejects any vehicle claims involving unauthorized commercial racing. All premium plan members must report any vehicular accident within 48 hours to preserve eligibility."]
}

print("📦 Step 3: Ingesting Logs for Strategy B (Intelligent Semantic Chunking)...")
strategy_b_logs = {
    "user_input": ["What are the rules regarding commercial racing and when must an accident be reported?"],
    "contexts": [["Our company premium insurance plan rejects any vehicle claims involving unauthorized commercial racing. All premium plan members must report any vehicular accident within 48 hours to preserve eligibility."]], 
    "response": ["Claims involving commercial racing are rejected, and you must report any accident within 24 hours."], 
    "reference": ["Our company premium insurance plan rejects any vehicle claims involving unauthorized commercial racing. All premium plan members must report any vehicular accident within 48 hours to preserve eligibility."]
}

# Convert raw dictionaries into HuggingFace Dataset formats for the RAGAS engine
dataset_a = Dataset.from_dict(strategy_a_logs)
dataset_b = Dataset.from_dict(strategy_b_logs)

print("📊 Step 4: Running RAGAS Evaluation on Strategy A...")
report_a = evaluate(dataset=dataset_a, metrics=[faithfulness, context_recall], llm=evaluator_llm, embeddings=evaluator_embeddings)
df_a = report_a.to_pandas()

print("📊 Step 5: Running RAGAS Evaluation on Strategy B...")
report_b = evaluate(dataset=dataset_b, metrics=[faithfulness, context_recall], llm=evaluator_llm, embeddings=evaluator_embeddings)
df_b = report_b.to_pandas()

print("\n=========== 🏁 HEAD-TO-HEAD SCOREBOARD ===========\n")
print(f"📐 STRATEGY A (Character): Faithfulness: {df_a['faithfulness'].values[0]:.2f} | Context Recall: {df_a['context_recall'].values[0]:.2f}")
print(f"🧠 STRATEGY B (Semantic):  Faithfulness: {df_b['faithfulness'].values[0]:.2f} | Context Recall: {df_b['context_recall'].values[0]:.2f}\n")
print("==================================================\n")

🔄 Step 1: Initializing the AI Judge (GPT-4o)...
📦 Step 2: Ingesting Logs for Strategy A (Basic Fixed Character Chunking)...
📦 Step 3: Ingesting Logs for Strategy B (Intelligent Semantic Chunking)...
📊 Step 4: Running RAGAS Evaluation on Strategy A...


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

📊 Step 5: Running RAGAS Evaluation on Strategy B...


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]


=========== 🏁 HEAD-TO-HEAD SCOREBOARD ===========

📐 STRATEGY A (Character): Faithfulness: 1.00 | Context Recall: 0.50
🧠 STRATEGY B (Semantic):  Faithfulness: 0.50 | Context Recall: 1.00


